In [ ]:
#Mounting drive for data
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#IMD Data
path1 = '/content/gdrive/MyDrive/GAIAData/80-20/TmaxData_2010_2014.xlsx'
IMD_Nasa = pd.read_excel(path1)
IMD_Nasa

,Date,Latitude,Longitude,IMD,NASA
0,2010-01-01,9.5,77.5,30.670000,29.19
1,2010-01-02,9.5,77.5,30.670000,28.98
2,2010-01-03,9.5,77.5,30.600000,28.71
3,2010-01-04,9.5,77.5,30.320000,29.55
4,2010-01-05,9.5,77.5,30.469999,30.91
...,...,...,...,...,...
18245,2014-12-27,11.5,79.5,28.343775,26.24
18246,2014-12-28,11.5,79.5,27.318331,26.23
18247,2014-12-29,11.5,79.5,28.781235,27.74
18248,2014-12-30,11.5,79.5,30.213234,29.59


In [ ]:
df = pd.DataFrame(IMD_Nasa)
df.dtypes

Date         datetime64[ns]
Latitude            float64
Longitude           float64
IMD                 float64
NASA                float64
dtype: object

In [ ]:
results = []

In [ ]:
def adjust_imd_values(df):

    results = []  # Store the adjusted IMD values here
    for index, row in df.iterrows():
        # Extracting current row details

        lat, lon, current_imd, date = row['Latitude'], row['Longitude'], row['IMD'], row['Date']

        # Initialize target IMD to NaN (indicating no match found initially)
        target_imd = np.nan

        # Conditions based on latitude and longitude
        if 9.5 <= lat < 11.5:
          target_row = df[(df['Latitude'] == lat + 1) & (df['Date'] == date)]
        elif lat == 11.5 and 78.5 <= lon < 79.5:
          target_row = df[(df['Longitude'] == lon + 1) & (df['Date'] == date) & (df['Latitude'] == lat)]
        elif lat == 11.5 and lon == 79.5:
          target_row = df[(df['Longitude'] == lon - 1) & (df['Date'] == date) & (df['Latitude'] == lat)]

        if not target_row.empty:
            # If a matching row is found, take its IMD value for subtraction
            target_imd = target_row.iloc[0]['IMD']
        # Perform the subtraction if a matching row was found; otherwise, leave as NaN or current IMD
        result = current_imd - target_imd if not pd.isnull(target_imd) else np.nan
        results.append(result)

    # Add the results to the DataFrame as a new column
    df['error'] = results

# Apply the function to adjust IMD values
adjust_imd_values(df)
df['error'] = df['error'].abs()

# Display the modified DataFrame
df

,Date,Latitude,Longitude,IMD,NASA,error
0,2010-01-01,9.5,77.5,30.670000,29.19,1.879999
1,2010-01-02,9.5,77.5,30.670000,28.98,2.180000
2,2010-01-03,9.5,77.5,30.600000,28.71,1.850000
3,2010-01-04,9.5,77.5,30.320000,29.55,2.160000
4,2010-01-05,9.5,77.5,30.469999,30.91,2.459999
...,...,...,...,...,...,...
18245,2014-12-27,11.5,79.5,28.343775,26.24,0.851707
18246,2014-12-28,11.5,79.5,27.318331,26.23,0.638184
18247,2014-12-29,11.5,79.5,28.781235,27.74,0.771967
18248,2014-12-30,11.5,79.5,30.213234,29.59,0.848019


In [ ]:
error_column = df['error']

# Calculate error metrics
mae = np.mean(np.abs(error_column))
mse = np.mean(error_column**2)
rmse = np.sqrt(mse)
std_deviation = np.std(error_column)

# Print calculated error metrics
print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Standard Deviation of Errors:", std_deviation)

Mean Absolute Error (MAE): 2.3964127245329574
Mean Squared Error (MSE): 10.978652914049167
Root Mean Squared Error (RMSE): 3.3134050332021237
Standard Deviation of Errors: 2.2881999405090667
